# Phase 05 - Feature Engineering & Preprocessing

This notebook uses reusable project modules. Its partition and transformations are an illustrative sample workflow only; it does not create the final modeling split, train a model, or establish the Phase 08 benign anomaly baseline.

## 1. Scope and environment

Install the project requirements before running. The sample loader reads a bounded number of rows from each Parquet file rather than materializing the 2.83 million row dataset.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from sklearn.model_selection import train_test_split

from intrusion_intelligence.config import load_config
from intrusion_intelligence.feature_engineering import (
    DERIVED_FEATURES, FeatureContract, FlowFeatureEngineer,
    separate_features_target_metadata,
)
from intrusion_intelligence.preprocessing.artifacts import load_preprocessor, save_preprocessor
from intrusion_intelligence.preprocessing.class_imbalance import training_class_distribution
from intrusion_intelligence.preprocessing.config import load_preprocessing_config
from intrusion_intelligence.preprocessing.pipeline import build_preprocessor
from intrusion_intelligence.preprocessing.storage import save_transformed_matrix

config = load_config()
contract = FeatureContract.from_file(config.project_root / 'configs' / 'feature_schema.json')
preprocessing_config = load_preprocessing_config(config.project_root / 'configs' / 'preprocessing.toml')
contract.dataset_name, len(contract.predictive_features), contract.numeric_features[:5]

## 2. Inspect the real cleaned schema and load a bounded sample

The feature contract was generated from the Phase 04 manifest and feature policy. Read the schema and sample a maximum of 1,000 rows per source file using Parquet batches.

In [ ]:
output_root = config.paths.processed_data / 'CIC-IDS2017-phase04-v2'
cleaned_root = output_root / 'cleaned'
parquet_files = sorted(cleaned_root.rglob('*.parquet'))
if not parquet_files:
    raise FileNotFoundError(f'No Phase 04 Parquet files under {cleaned_root}')

available_metadata = [name for name in contract.metadata_fields if name in pq.ParquetFile(parquet_files[0]).schema_arrow.names]
sample_columns = [*contract.predictive_features, contract.target_name, *available_metadata]
def read_bounded_sample(path, row_limit=1000):
    parquet = pq.ParquetFile(path)
    batches = parquet.iter_batches(batch_size=row_limit, columns=sample_columns)
    first_batch = next(batches, None)
    return pd.DataFrame(columns=sample_columns) if first_batch is None else first_batch.to_pandas()

sample = pd.concat([read_bounded_sample(path) for path in parquet_files], ignore_index=True)
schema_summary = pd.DataFrame([{'name': feature.name, 'type': feature.feature_type, 'included': feature.included, 'source': feature.source_column} for feature in contract.features])
len(parquet_files), len(sample), schema_summary.head()

## 3. Separate X, y, and metadata

Selection is explicit. The schema helper rejects unrecognized columns, validates the target, and keeps row indexes aligned across features, labels, and metadata.

In [ ]:
X, y, metadata = separate_features_target_metadata(sample, contract)
X.shape, y.shape, metadata.shape, list(X.columns) == list(contract.predictive_features)

## 4. Exclude target and leakage fields

`label` is returned as y. Source labels, file/row provenance, record IDs, and any recognized identifiers or timestamp fields are returned separately as metadata. They cannot be passed through the feature-only validator.

In [ ]:
prohibited = {contract.target_name, *contract.metadata_fields, *contract.excluded_fields}
assert not prohibited.intersection(X.columns)
{'features': len(X.columns), 'metadata_fields_present': list(metadata.columns), 'target': y.name}

## 5. Numerical and categorical feature types

Types come from the contract, not numeric-looking storage. The actual Phase 04 schema has 78 numeric candidates and no categorical candidates; destination port is retained as numeric. Protocol remains a supported categorical type if a later reviewed schema adds it.

In [ ]:
type_summary = {'numeric': list(contract.numeric_features), 'categorical': list(contract.categorical_features), 'unavailable_candidates': json.loads((config.project_root / 'configs' / 'feature_schema.json').read_text(encoding='utf-8'))['unavailable_predictive_candidates']}
{name: len(values) if isinstance(values, list) else values for name, values in type_summary.items()}

## 6. Optional derived feature definitions

The layer can add aggregate packet/byte counts and directional proportions. Ratios become missing for a zero denominator, missing source, or negative directional count. They are disabled by default because they are algebraically related to existing fields and require later ablation.

In [ ]:
pd.DataFrame([{'name': name, 'formula': definition['formula'], 'purpose': definition['purpose'], 'missing': definition['missing_behavior'], 'zero': definition['zero_behavior'], 'redundancy': definition['redundancy']} for name, definition in DERIVED_FEATURES.items()])

## 7. Demonstrate optional flow feature engineering

This demonstration transforms only a small sample. It uses the reusable module and does not use y or metadata.

In [ ]:
derived_demo = FlowFeatureEngineer(('total_packets', 'forward_packet_proportion'))
derived_preview = derived_demo.fit_transform(X.head(1000))
derived_preview[['total_packets', 'forward_packet_proportion']].head()

## 8. Inspect missingness before fitting

Phase 04 retained missing predictive measurements as nulls. The median imputer below learns only from X_train; no imputation statistic is calculated from the complete dataset or the held-out rows.

In [ ]:
missing_review = pd.DataFrame({'missing_rows': X.isna().sum(), 'missing_fraction': X.isna().mean()})
missing_review.loc[missing_review['missing_rows'] > 0].sort_values('missing_rows', ascending=False).head(20)

## 9. Categorical encoding and unknown categories

There are no categorical columns in the current cleaned extract, so protocol encoding is demonstrated with an explicitly synthetic fixture. Numeric protocol codes are stringified as categories, missing categories use `__MISSING__`, and unknown inference values are ignored without changing output width.

In [ ]:
protocol_contract = FeatureContract.from_dict({'schema_version': 'demo-1.0', 'dataset_name': 'synthetic-protocol-demo', 'target': {'name': 'label'}, 'features': [{'name': 'flow_duration', 'type': 'numeric', 'included': True}, {'name': 'protocol', 'type': 'categorical', 'included': True}], 'leakage_policy': {'metadata_fields': [], 'excluded_fields': []}})
protocol_train = pd.DataFrame({'flow_duration': [1.0, 2.0, np.nan], 'protocol': [6, 17, np.nan]})
protocol_pipe = build_preprocessor('tree', contract=protocol_contract).fit(protocol_train)
protocol_unknown = protocol_pipe.transform(pd.DataFrame({'flow_duration': [3.0], 'protocol': [99]}))
protocol_pipe.get_feature_names_out(), protocol_unknown.shape

## 10. Compare scaling profiles

Linear models use median imputation and StandardScaler; tree models use median imputation without scaling; anomaly preprocessing uses median imputation and RobustScaler. These are configurable starting points, not claims of optimality. Categorical one-hot outputs are not scaled.

In [ ]:
pd.DataFrame([{'profile': name, **settings, 'numeric_imputation': preprocessing_config['imputation']['numeric_strategy'], 'categorical_imputation': preprocessing_config['imputation']['categorical_strategy']} for name, settings in preprocessing_config['profiles'].items()])

## 11. Optional train-fitted feature selection

VarianceThreshold is available and disabled by default. EDA found eight full-dataset constant fields, but the selector still fits only on training rows so held-out data cannot affect selection. Correlation-based removal is not enabled.

In [ ]:
preprocessing_config['feature_selection'], len(json.loads((config.project_root / 'reports' / 'eda' / 'CIC-IDS2017_eda.json').read_text(encoding='utf-8'))['feature_quality']['constant_features_exact_full'])

## 12. Create a clearly illustrative train/holdout partition

This random partition is for checking preprocessing consistency on the bounded sample only. It is not the project's final split or an evaluation design; file/label concentration and duplicate groups require later group-aware analysis.

In [ ]:
train_indices, holdout_indices = train_test_split(np.arange(len(X)), test_size=0.2, random_state=preprocessing_config['random_seed'])
X_train, X_holdout = X.iloc[train_indices], X.iloc[holdout_indices]
y_train, y_holdout = y.iloc[train_indices], y.iloc[holdout_indices]
metadata_holdout = metadata.iloc[holdout_indices]
X_train.shape, X_holdout.shape, y_train.shape, metadata_holdout.shape

## 13. Fit the supervised preprocessing profile on training features only

No target is passed to `fit`. Derived features are explicitly enabled for this demonstration; the production default remains off.

In [ ]:
supervised_preprocessor = build_preprocessor('linear', contract=contract, derived_features=('total_packets', 'forward_packet_proportion'))
X_train_transformed = supervised_preprocessor.fit_transform(X_train)
X_holdout_transformed = supervised_preprocessor.transform(X_holdout)
X_train_transformed.shape, X_holdout_transformed.shape, len(supervised_preprocessor.get_feature_names_out())

## 14. Verify ordering and held-out consistency

Transform uses the fitted imputer, encoder, scaler, and selector. It does not call fit or update learned statistics. Batches use this same fitted instance.

In [ ]:
names_before = supervised_preprocessor.get_feature_names_out().tolist()
medians_before = supervised_preprocessor.numeric_imputation_statistics.copy()
batch_transforms = list(supervised_preprocessor.transform_batches([X_holdout.iloc[:500], X_holdout.iloc[500:]]))
assert names_before == supervised_preprocessor.get_feature_names_out().tolist()
assert medians_before == supervised_preprocessor.numeric_imputation_statistics
assert sum(block.shape[0] for block in batch_transforms) == len(X_holdout)
{'ordered_features': len(names_before), 'training_medians': medians_before, 'scaler_parameters': supervised_preprocessor.scaling_statistics()}

## 15. Compare independent supervised and anomaly profiles

The anomaly profile is a separate configuration and must be fitted later on the Phase 08 permitted benign training baseline. This notebook does not fit it on the mixed-label sample.

In [ ]:
linear_instance = build_preprocessor('linear', contract=contract)
tree_instance = build_preprocessor('tree', contract=contract)
anomaly_instance = build_preprocessor('anomaly', contract=contract)
[(item.profile, item.config['profiles'][item.profile]['scaler']) for item in (linear_instance, tree_instance, anomaly_instance)], len({id(linear_instance), id(tree_instance), id(anomaly_instance)}) == 3

## 16. Training-only class distribution and optional class weights

This helper accepts `y_train` only. It provides counts and optional balanced weights without applying resampling; validation and holdout labels are not passed.

In [ ]:
training_class_distribution(y_train, class_weight='balanced')

## 17. Save and reload the fitted preprocessing artifact

Artifacts are versioned by profile and version and will not overwrite an existing path. Use a new version for a changed schema or fit. The JSON sidecar contains schema/configuration/statistics metadata, not raw traffic rows.

In [ ]:
artifact_root = config.paths.model_artifacts / 'preprocessors'
artifact_directory = save_preprocessor(supervised_preprocessor, artifact_root, version='1.0.0')
loaded_preprocessor = load_preprocessor(artifact_directory)
reloaded_holdout = loaded_preprocessor.transform(X_holdout)
metadata_path = artifact_directory / 'metadata.json'
artifact_metadata = json.loads(metadata_path.read_text(encoding='utf-8'))
artifact_directory, reloaded_holdout.shape, artifact_metadata.keys()

## 18. Persist an aligned transformed sample when needed

The storage helper uses NPY for dense matrices or SciPy NPZ for sparse matrices and writes target plus row metadata to Parquet in matching order. This cell stores only the illustrative held-out sample.

In [ ]:
sample_matrix_directory = config.paths.processed_data / 'phase05-illustrative-holdout-v1'
saved_sample = save_transformed_matrix(
    X_holdout_transformed, y_holdout, metadata_holdout,
    supervised_preprocessor.get_feature_names_out().tolist(),
    sample_matrix_directory, target_name=contract.target_name,
)
json.loads((saved_sample / 'manifest.json').read_text(encoding='utf-8'))

## 19. Findings and limitations

The actual cleaned schema supports numeric flow features only; protocol and source port are absent. The sample is bounded and transformations were fitted only on its illustrative training rows. No full dataset was transformed, no final split or model was created, no anomaly baseline was fitted, and feature choices have not been validated by model experiments.